### All days of the challange:

* [Day 1: Handling missing values](https://www.kaggle.com/rtatman/data-cleaning-challenge-handling-missing-values)
* [Day 2: Scaling and normalization](https://www.kaggle.com/rtatman/data-cleaning-challenge-scale-and-normalize-data)
* [Day 3: Parsing dates](https://www.kaggle.com/rtatman/data-cleaning-challenge-parsing-dates/)
* [Day 4: Character encodings](https://www.kaggle.com/rtatman/data-cleaning-challenge-character-encodings/)
* [Day 5: Inconsistent Data Entry](https://www.kaggle.com/rtatman/data-cleaning-challenge-inconsistent-data-entry/)
___
Welcome to day 4 of the 5-Day Data Challenge! Today, we're going to be working with different character encodings. To get started, click the blue "Fork Notebook" button in the upper, right hand corner. This will create a private copy of this notebook that you can edit and play with. Once you're finished with the exercises, you can choose to make your notebook public to share with others. :)

> **Your turn!** As we work through this notebook, you'll see some notebook cells (a block of either code or text) that has "Your Turn!" written in it. These are exercises for you to do to help cement your understanding of the concepts we're talking about. Once you've written the code to answer a specific question, you can run the code by clicking inside the cell (box with code in it) with the code you want to run and then hit CTRL + ENTER (CMD + ENTER on a Mac). You can also click in a cell and then click on the right "play" arrow to the left of the code. If you want to run all the code in your notebook, you can use the double, "fast forward" arrows at the bottom of the notebook editor.

Here's what we're going to do today:

* [Get our environment set up](#Get-our-environment-set-up)
* [What are encodings?](#What-are-encodings?)
* [Reading in files with encoding problems](#Reading-in-files-with-encoding-problems)
* [Saving your files with UTF-8 encoding](#Saving-your-files-with-UTF-8-encoding)

Let's get started!

# Get our environment set up
________

The first thing we'll need to do is load in the libraries we'll be using. Not our datasets, though: we'll get to those later!

> **Important!** Make sure you run this cell yourself or the rest of your code won't work!

In [52]:
# modules we'll use
import pandas as pd
import numpy as np

# helpful character encoding module
import chardet
print(f"Version de chardet : {chardet.__version__}")

# set seed for reproducibility
np.random.seed(0)

Version de chardet : 3.0.4


Now we're ready to work with some character encodings! (If you like, you can add a code cell here and take this opportunity to take a look at some of the data.)

# What are encodings?
____

Character encodings are specific sets of rules for mapping from raw binary byte strings (that look like this: 0110100001101001) to characters that make up human-readable text (like "hi"). There are many different encodings, and if you tried to read in text with a different encoding that the one it was originally written in, you ended up with scrambled text called "mojibake" (said like mo-gee-bah-kay). Here's an example of mojibake:

æ–‡å—åŒ–ã??

You might also end up with a "unknown" characters. There are what gets printed when there's no mapping between a particular byte and a character in the encoding you're using to read your byte string in and they look like this:

����������

Character encoding mismatches are less common today than they used to be, but it's definitely still a problem. There are lots of different character encodings, but the main one you need to know is UTF-8.

> UTF-8 is **the** standard text encoding. UTF-8 is the default encoding for Python 3 source files (unless another source encoding is declared), and ideally your data files should use a known, explicit encoding such as UTF-8. and, ideally, all your data should be as well. It's when things aren't in UTF-8 that you run into trouble.

It was pretty hard to deal with encodings in Python 2, but thankfully in Python 3 it's a lot simpler. (Kaggle Kernels only use Python 3.) There are two main data types you'll encounter when working with text in Python 3. One is is the string, which is what text is by default.

In [53]:
# start with a string
before = "This is the euro symbol: €"

# check to see what datatype it is
type(before)

str

The other data is the [bytes](https://docs.python.org/3.1/library/functions.html#bytes) data type, which is a sequence of integers. You can convert a string into bytes by specifying which encoding it's in:

In [54]:
# encode it to a different encoding, replacing characters that raise errors
after = before.encode("utf-8", errors = "replace")

# check the type
type(after)

bytes

If you look at a bytes object, you'll see a `b` in front of it. Its representation shows printable ASCII bytes directly and uses escape sequences such as `\xNN` for other byte values. Here `\xe2\x82\xac` is simply the escaped representation of the three UTF-8 bytes encoding the euro sign; it is not mojibake.

In [55]:
# take a look at what the bytes look like
after

b'This is the euro symbol: \xe2\x82\xac'

When we convert our bytes back to a string with the correct encoding, we can see that our text is all there correctly, which is great! :)

In [56]:
# convert it back to utf-8
print(after.decode("utf-8"))

This is the euro symbol: €


However, when we try to use a different encoding to map our bytes into a string,, we get an error. This is because the encoding we're trying to use doesn't know what to do with the bytes we're trying to pass it. You need to tell Python the encoding that the byte string is actually supposed to be in.

> You can think of different encodings as different ways of recording music. You can record the same music on a CD, cassette tape or 8-track. While the music may sound more-or-less the same, you need to use the right equipment to play the music from each recording format. The correct decoder is like a cassette player or a cd player. If you try to play a cassette in a CD player, it just won't work. 

In [57]:
# try to decode our bytes with the ascii encoding
try:
    print(after.decode("ascii"))
except UnicodeDecodeError as err:
    # Erreur attendue : l'octet 0xe2 (début du « € » en UTF-8) n'existe pas en ASCII
    print(f"UnicodeDecodeError (attendue) : {err}")

UnicodeDecodeError (attendue) : 'ascii' codec can't decode byte 0xe2 in position 25: ordinal not in range(128)


**Remarque :** cette cellule et la cellule « try to read in a file not in UTF-8 » provoquent volontairement une `UnicodeDecodeError`. Elles sont encadrées par un bloc `try / except` : le message d'erreur reste visible, mais il n'interrompt plus l'exécution complète du notebook.

We can also run into trouble if we try to use the wrong encoding to map from a string to bytes. Like I said earlier, strings contain Unicode text in Python 3; they are not UTF-8 bytes until they are explicitly encoded, so if we try to treat them like they were in another encoding we'll create problems. 

For example, if we try to convert a string to bytes for ascii using encode(), we can ask for the bytes to be what they would be if the text was in ASCII. Since our text isn't in ASCII, though, there will be some characters it can't handle. We can automatically replace the characters that ASCII can't handle. If we do that, however, any characters not in ASCII will just be replaced with the unknown character. Then, when we convert the bytes back to a string, the character will be replaced with the unknown character. The dangerous part about this is that there's not way to tell which character it *should* have been. That means we may have just made our data unusable!

In [58]:
# start with a string
before = "This is the euro symbol: €"

# encode it to a different encoding, replacing characters that raise errors
after = before.encode("ascii", errors = "replace")

# convert it back to utf-8
print(after.decode("ascii"))

# We've lost the original underlying byte string! It's been 
# replaced with the underlying byte string for the unknown character :(

This is the euro symbol: ?


This is bad and we want to avoid doing it! It's far better to convert all our text to UTF-8 as soon as we can and keep it in that encoding. The best time to convert non UTF-8 input into UTF-8  is when you read in files, which we'll talk about next.

First, however, try converting between bytes and strings with different encodings and see what happens. Notice what this does to your text. Would you want this to happen to data you were trying to analyze?

In [59]:
# Your turn! Try encoding and decoding different symbols to ASCII and
# see what happens. I'd recommend $, #, 你好 and नमस्ते but feel free to
# try other characters. What happens? When would this cause problems?
samples = ["$", "#", "é", "€", "你好", "नमस्ते"]

for text in samples:
    code_points = [f"U+{ord(c):04X}" for c in text]
    utf8_bytes = text.encode("utf-8")
    ascii_roundtrip = text.encode("ascii", errors="replace").decode("ascii")
    print(f"{text!r}")
    print(f"   points de code Unicode : {code_points}")
    print(f"   UTF-8 : {len(utf8_bytes)} octet(s) → {utf8_bytes}")
    print(f"   aller-retour par l'ASCII : {ascii_roundtrip!r}")

# Le mojibake classique : des octets UTF-8 relus comme du Windows-1252
print(f"\n« café » encodé en UTF-8 puis relu en Windows-1252 : {'café'.encode('utf-8').decode('windows-1252')!r}")

'$'
   points de code Unicode : ['U+0024']
   UTF-8 : 1 octet(s) → b'$'
   aller-retour par l'ASCII : '$'
'#'
   points de code Unicode : ['U+0023']
   UTF-8 : 1 octet(s) → b'#'
   aller-retour par l'ASCII : '#'
'é'
   points de code Unicode : ['U+00E9']
   UTF-8 : 2 octet(s) → b'\xc3\xa9'
   aller-retour par l'ASCII : '?'
'€'
   points de code Unicode : ['U+20AC']
   UTF-8 : 3 octet(s) → b'\xe2\x82\xac'
   aller-retour par l'ASCII : '?'
'你好'
   points de code Unicode : ['U+4F60', 'U+597D']
   UTF-8 : 6 octet(s) → b'\xe4\xbd\xa0\xe5\xa5\xbd'
   aller-retour par l'ASCII : '??'
'नमस्ते'
   points de code Unicode : ['U+0928', 'U+092E', 'U+0938', 'U+094D', 'U+0924', 'U+0947']
   UTF-8 : 18 octet(s) → b'\xe0\xa4\xa8\xe0\xa4\xae\xe0\xa4\xb8\xe0\xa5\x8d\xe0\xa4\xa4\xe0\xa5\x87'
   aller-retour par l'ASCII : '??????'

« café » encodé en UTF-8 puis relu en Windows-1252 : 'cafÃ©'


**Réponse :**

**1. UTF-8 est à longueur variable.** `$` et `#` occupent 1 octet, `é` 2 octets, `€` 3 octets, et chaque point de code chinois ou devanagari utilisé ici 3 octets. Le nombre d'octets dépend du point de code Unicode :

| Points de code | Octets | Motif binaire |
|---|---|---|
| U+0000 à U+007F | 1 | `0xxxxxxx` |
| U+0080 à U+07FF | 2 | `110xxxxx 10xxxxxx` |
| U+0800 à U+D7FF et U+E000 à U+FFFF | 3 | `1110xxxx 10xxxxxx 10xxxxxx` |
| U+10000 à U+10FFFF | 4 | `11110xxx 10xxxxxx 10xxxxxx 10xxxxxx` |

La plage U+D800 à U+DFFF est réservée aux surrogates UTF-16 : ces valeurs ne sont pas des valeurs scalaires Unicode et ne sont pas encodées en UTF-8.

Exemple : `€` = U+20AC = $0010\ 0000\ 1010\ 1100_2$. On répartit ces 16 bits en $4 + 6 + 6$ dans le motif à 3 octets : `1110 0010`, `10 000010`, `10 101100`, soit exactement `E2 82 AC`, les octets affichés ci-dessus.

**2. ASCII est inclus dans UTF-8.** Les caractères ASCII (U+0000 à U+007F) ont le même octet unique dans les deux encodages. C'est pourquoi `$` et `#` survivent à l'aller-retour.

**3. L'aller-retour par l'ASCII détruit l'information.** Tout caractère hors ASCII devient `?`. `€`, `é` et `你` donnent le même résultat, la perte est donc **irréversible** : on ne peut plus retrouver le texte d'origine. En pratique, cela corrompt les noms de personnes et de lieux, empêche des jointures entre tables (« Müller » ne correspond plus à « M?ller ») et peut créer de faux doublons.

**4. Un « caractère » n'est pas nécessairement un point de code.** `नमस्ते` contient 6 points de code : des signes combinants comme le virama (U+094D) ou la voyelle U+0947 se combinent avec d'autres signes. `len()` compte des points de code, pas nécessairement les caractères perçus par le lecteur.

**5. Le mojibake.** « café » encodé en UTF-8 donne les octets `63 61 66 C3 A9`. Relus en Windows-1252, `C3` devient « Ã » et `A9` devient « © », d'où « cafÃ© ». Ce n'est pas nécessairement une perte : si les octets d'origine sont restés intacts, il suffit de les décoder avec le bon encodage.

# Reading in files with encoding problems
___

Most files you'll encounter will probably be encoded with UTF-8. This is what Python expects by default, so most of the time you won't run into problems. However, sometimes you'll get an error like this: 

In [60]:
# try to read in a file not in UTF-8
try:
    kickstarter_2016 = pd.read_csv("../input/kickstarter-projects/ks-projects-201612.csv")
except UnicodeDecodeError as err:
    # Erreur attendue : le fichier n'est pas encodé en UTF-8
    print(f"UnicodeDecodeError (attendue) : {err}")

UnicodeDecodeError (attendue) : 'utf-8' codec can't decode byte 0x99 in position 11: invalid start byte


Notice that we get the same `UnicodeDecodeError` we got when we tried to decode UTF-8 bytes as if they were ASCII! This tells us that this file isn't actually UTF-8. We don't know what encoding it actually *is* though. One way to figure it out is to try and test a bunch of different character encodings and see if any of them work. A better way, though, is to use the chardet module to try and automatically guess what the right encoding is. It's not 100% guaranteed to be right, but it's usually faster than just trying to guess.

I'm going to just look at the first ten thousand bytes of this file. This is usually enough for a good guess about what the encoding is and is much faster than trying to look at the whole file. (Especially with a  large file this can be very slow.) Another reason to just look at the first part of the file is that  we can see by looking at the error message that the first problem is the 11th character. So we probably only need to look at the first little bit of the file to figure out what's going on.

In [61]:
# look at the first ten thousand bytes to guess the character encoding
with open("../input/kickstarter-projects/ks-projects-201801.csv", 'rb') as rawdata:
    result = chardet.detect(rawdata.read(10000))

# check what the character encoding might be
print(result)

{'encoding': 'Windows-1252', 'confidence': 0.73, 'language': ''}


In [62]:
# Bonus : la cellule ci-dessus analyse le fichier 201801, alors que c'est le fichier 201612
# qui provoque l'erreur. On analyse donc le bon fichier, avec deux tailles d'échantillon.
ks_2016_path = "../input/kickstarter-projects/ks-projects-201612.csv"
with open(ks_2016_path, 'rb') as rawdata:
    head_bytes = rawdata.read(100000)

for n_bytes in [10000, 100000]:
    guess = chardet.detect(head_bytes[:n_bytes])
    print(f"chardet sur les {n_bytes} premiers octets de 201612 : "
          f"{guess['encoding']} (confiance {guess['confidence']:.2f})")

chardet sur les 10000 premiers octets de 201612 : Windows-1252 (confiance 0.73)
chardet sur les 100000 premiers octets de 201612 : Windows-1252 (confiance 0.73)


**Analyse.** L'erreur indique l'octet `0x99`, invalide en début de caractère UTF-8 : les octets de `0x80` à `0xBF` ont la forme `10xxxxxx` et ne peuvent être que des octets de **continuation**. En Windows-1252, `0x99` est le symbole « ™ ».

**Une incohérence dans la cellule d'origine.** Elle analyse le fichier `201801` alors que c'est le fichier `201612` qui provoque l'erreur. La cellule bonus analyse le bon fichier : même verdict, Windows-1252.

**Pour aller plus loin : que vaut le « 0,73 » ?** On obtient exactement **0,73** dans les trois analyses affichées. Ce n'est pas une coïncidence : dans `chardet`, le détecteur de la famille Latin-1 / Windows-1252 **multiplie volontairement sa confiance par 0,73**, pour laisser la priorité aux détecteurs plus spécifiques. Ce nombre n'est donc pas une probabilité, mais un score heuristique plafonné.

D'ailleurs, le fichier `201801` a été lu **sans erreur en UTF-8** au notebook 2, alors que `chardet` le classe ici en Windows-1252. Le diagnostic de `chardet` est une hypothèse à vérifier, pas une preuve : en Mac Roman, `0x99` vaudrait « ô ». Le choix de Windows-1252 n'est donc pas vérifié ici par le sens, contrairement à ce qui est fait plus bas pour `PoliceKillingsUS.csv`.

So chardet is 73%  confidence that the right encoding is "Windows-1252". Let's see if that's correct:

In [63]:
# read in the file with the encoding detected by chardet
kickstarter_2016 = pd.read_csv("../input/kickstarter-projects/ks-projects-201612.csv", encoding='Windows-1252')

# look at the first few lines
kickstarter_2016.head()

/opt/conda/lib/python3.6/site-packages/IPython/core/interactiveshell.py:2698: DtypeWarning: Columns (13,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


,ID,name,category,main_category,currency,deadline,goal,launched,pledged,state,backers,country,usd pledged,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16
0,1000002330,The Songs of Adelaide & Abullah,Poetry,Publishing,GBP,2015-10-09 11:36:00,1000,2015-08-11 12:12:28,0,failed,0,GB,0,NaN,NaN,NaN,NaN
1,1000004038,Where is Hank?,Narrative Film,Film & Video,USD,2013-02-26 00:20:50,45000,2013-01-12 00:20:50,220,failed,3,US,220,NaN,NaN,NaN,NaN
2,1000007540,ToshiCapital Rekordz Needs Help to Complete Album,Music,Music,USD,2012-04-16 04:24:11,5000,2012-03-17 03:24:11,1,failed,1,US,1,NaN,NaN,NaN,NaN
3,1000011046,Community Film Project: The Art of Neighborhoo...,Film & Video,Film & Video,USD,2015-08-29 01:00:00,19500,2015-07-04 08:35:03,1283,canceled,14,US,1283,NaN,NaN,NaN,NaN
4,1000014025,Monarch Espresso Bar,Restaurants,Food,USD,2016-04-01 13:38:27,50000,2016-02-26 13:38:27,52375,successful,224,US,52375,NaN,NaN,NaN,NaN


The file reads without an encoding error and the first few rows look plausible, but that alone does not prove that Windows-1252 is the correct encoding.

> **What if the encoding chardet guesses isn't right?** Since chardet is basically just a fancy guesser, sometimes it will guess the wrong encoding. One thing you can try is looking at more or less of the file and seeing if you get a different result and then try that.

In [64]:
# Your Turn! Trying to read in this file gives you an error. Figure out
# what the correct encoding should be and read in the file. :)
police_path = "../input/fatal-police-shootings-in-the-us/PoliceKillingsUS.csv"

with open(police_path, 'rb') as rawdata:
    raw_bytes = rawdata.read()
print(f"Taille du fichier : {len(raw_bytes)} octets")

# 1) Pourquoi l'UTF-8 échoue : on localise le premier octet invalide
try:
    raw_bytes.decode('utf-8')
except UnicodeDecodeError as err:
    print(f"UTF-8 invalide à la position {err.start} : octet {raw_bytes[err.start:err.start + 1]}")
    print(f"Contexte : {raw_bytes[max(err.start - 30, 0):err.start + 10]}")

# 2) Le diagnostic de chardet dépend de la taille de l'échantillon analysé
for n_bytes in [10000, 100000, len(raw_bytes)]:
    guess = chardet.detect(raw_bytes[:n_bytes])
    print(f"chardet sur {n_bytes} octets : {guess['encoding']} (confiance {guess['confidence']:.2f})")

# 3) Lecture avec l'encodage proposé par chardet (examiné en détail dans la cellule suivante)
encoding = chardet.detect(raw_bytes)['encoding']
police_killings = pd.read_csv(police_path, encoding=encoding)
print(f"\nEncodage proposé par chardet : {encoding} ; dimensions : {police_killings.shape}")

# Colonnes contenant des caractères non ASCII après décodage
text_cols = police_killings.select_dtypes(include='object').columns
non_ascii_counts = {col: police_killings[col].astype(str).str.contains(r'[^\x00-\x7F]').sum()
                    for col in text_cols}
print("Valeurs contenant des caractères non ASCII, par colonne :")
print({col: n for col, n in non_ascii_counts.items() if n > 0})

Taille du fichier : 227653 octets
UTF-8 invalide à la position 25272 : octet b'\x96'
Contexte : b'on,15/04/15,shot,gun,72,M,W,Ca\x96on City,C'
chardet sur 10000 octets : ascii (confiance 1.00)
chardet sur 100000 octets : Windows-1252 (confiance 0.73)
chardet sur 227653 octets : Windows-1252 (confiance 0.73)

Encodage proposé par chardet : Windows-1252 ; dimensions : (2535, 14)
Valeurs contenant des caractères non ASCII, par colonne :
{'city': 1}


In [65]:
# Analyse fine : chaque octet non ASCII du fichier, lu selon deux encodages candidats
import re

matches = list(re.finditer(rb'[\x80-\xff]', raw_bytes))
distinct_bytes = sorted(set(m.group() for m in matches))
print(f"Octets non ASCII dans le fichier : {len(matches)}")
print(f"Valeurs distinctes : {distinct_bytes}\n")

for m in matches:
    context = raw_bytes[max(m.start() - 20, 0):m.end() + 10]
    print(f"position {m.start():>6} | Windows-1252 : {context.decode('windows-1252', errors='replace')!r}")
    print(f"                | Mac Roman    : {context.decode('mac_roman')!r}")

Octets non ASCII dans le fichier : 1
Valeurs distinctes : [b'\x96']

position  25272 | Windows-1252 : '5,shot,gun,72,M,W,Ca–on City,CO'
                | Mac Roman    : '5,shot,gun,72,M,W,Cañon City,CO'


In [66]:
# Conclusion du diagnostic :
# plusieurs codecs Macintosh donnent ici « Cañon City » ;
# on retient Mac Roman comme hypothèse de travail.
police_killings = pd.read_csv(police_path, encoding='mac_roman')

has_non_ascii = police_killings['city'].str.contains(
    r'[^\x00-\x7F]',
    na=False
)

print(f"Encodage retenu : mac_roman ; dimensions : {police_killings.shape}")
print(
    f"Villes contenant des caractères non ASCII : "
    f"{police_killings.loc[has_non_ascii, 'city'].tolist()}"
)

Encodage retenu : mac_roman ; dimensions : (2535, 14)
Villes contenant des caractères non ASCII : ['Cañon City']


**Réponse : le décodage Mac Roman est cohérent avec le contenu attendu ; Windows-1252 produit ici un texte sémantiquement faux.**

**1. Le symptôme.** La lecture en UTF-8 échoue à la position 25 272 (indice compté à partir de 0) sur l'octet `0x96`, dans le nom de ville `Ca\x96on City`.

**2. Le piège de l'échantillon.** Sur les 10 000 premiers octets, `chardet` répond « ascii » avec une confiance de 1,00. C'est logique, puisque le seul octet problématique se trouve bien plus loin. Sur 100 000 octets ou sur le fichier entier, il répond Windows-1252 avec un score de 0,73.

**3. Le piège principal : décoder sans erreur ≠ décoder correctement.** L'analyse octet par octet montre que le fichier entier ne contient qu'**un seul octet non ASCII**. Lu en Windows-1252, `0x96` devient « – » (tiret demi-cadratin) : pandas lit le fichier sans erreur, mais la ville devient « Ca–on City ». Lu en **Mac Roman**, `0x96` devient « ñ » : on obtient **Cañon City**, le nom attendu. Windows-1252 est donc décodable, mais son interprétation est incohérente avec le contenu attendu.

**4. Pour aller plus loin : un problème d'identifiabilité.** Windows-1252 et Mac Roman décodent tous deux ce fichier sans exception ; les octets seuls ne permettent donc pas de trancher. `chardet` 3.0.4 ne disposait pas du détecteur MacRoman, ajouté en version 5.1.0. D'autres codecs Macintosh (`mac_iceland`, `mac_turkish`, `mac_croatian`…) donnent également « ñ » pour `0x96` et peuvent produire ici le même texte. Le seul octet `0x96` ne permet donc pas d'identifier de façon unique l'encodage exact, et il ne permet pas non plus d'inférer le système d'exploitation ou le logiciel ayant créé le fichier. La décision repose ici sur la **validation par le sens** du texte décodé.

#### Saving your files with UTF-8 encoding
___

Finally, once you've gone through all the trouble of getting your file into UTF-8, you'll probably want to keep it that way. The easiest way to do that is to save your files with UTF-8 encoding. The good news is, since UTF-8 is the standard encoding in Python, when you save a file it will be saved as UTF-8 by default:

In [67]:
# save our file (will be saved as UTF-8 by default!)
kickstarter_2016.to_csv("ks-projects-201801-utf8.csv")

Pretty easy, huh? :)

> If you haven't saved a file in a kernel before, you need to hit the commit & run button and wait for your notebook to finish running first before you can see or access the file you've saved out. If you don't see it at first, wait a couple minutes and it should show up. The files you save will be in the directory "../output/", and you can download them from your notebook.

In [68]:
# Your turn! Save out a version of the police_killings dataset with UTF-8 encoding 
output_path = "PoliceKillingsUS-utf8.csv"
police_killings.to_csv(output_path, index=False, encoding="utf-8")

# Vérification 1 : le fichier écrit est du UTF-8 strictement valide
with open(output_path, 'rb') as saved:
    saved_bytes = saved.read()
saved_bytes.decode('utf-8')
print("Le fichier sauvegardé est du UTF-8 valide.")

# Vérification 2 : le « ñ » est désormais codé sur 2 octets UTF-8 au lieu d'1 octet Mac Roman
print(f"« ñ » en Mac Roman : {'ñ'.encode('mac_roman')} ; en UTF-8 : {'ñ'.encode('utf-8')}")
print(f"« Cañon City » présent dans le fichier UTF-8 : {'Cañon City'.encode('utf-8') in saved_bytes}")

# Vérification 3 : relu sans préciser d'encodage (UTF-8 par défaut), le contenu textuel est identique
reloaded = pd.read_csv(output_path)
same_text = all((reloaded[col].fillna('') == police_killings[col].fillna('')).all() for col in ['name', 'city'])
print(f"Dimensions relues : {reloaded.shape} ; colonnes name et city identiques : {same_text}")

Le fichier sauvegardé est du UTF-8 valide.
« ñ » en Mac Roman : b'\x96' ; en UTF-8 : b'\xc3\xb1'
« Cañon City » présent dans le fichier UTF-8 : True
Dimensions relues : (2535, 14) ; colonnes name et city identiques : True


**Réponse : sauvegarde en UTF-8.** Le fichier écrit est du UTF-8 strictement valide. Le « ñ », codé sur un seul octet en Mac Roman (`0x96`), est désormais codé sur deux octets en UTF-8 (`C3 B1`). Il est relu correctement dans cette cellule ; tout outil configuré pour lire de l'UTF-8 pourra le décoder correctement.

Deux détails :
- `index=False` évite d'écrire l'index pandas comme une colonne supplémentaire, qui réapparaîtrait sous le nom `Unnamed: 0` à la relecture.
- La cellule de démonstration plus haut enregistre les données **2016** sous le nom `ks-projects-201801-utf8.csv`. Ce nom est trompeur ; il vaudrait mieux l'appeler `ks-projects-201612-utf8.csv`.

## Synthèse

| Étape | Constat | Choix et justification |
|---|---|---|
| Encoder et décoder | UTF-8 utilise 1 à 4 octets par valeur scalaire Unicode ; l'ASCII en est un sous-ensemble ; un passage par l'ASCII remplace tout le reste par `?` | Rester en UTF-8 de bout en bout : la perte par `?` est irréversible |
| Kickstarter 2016 | Octet `0x99` invalide en UTF-8 ; `chardet` propose Windows-1252 avec un score de 0,73 | Lecture en Windows-1252 (`0x99` = « ™ »), mais le sens ne permet pas ici de confirmer cet encodage |
| `PoliceKillingsUS` | Un seul octet non ASCII (`0x96`) ; Windows-1252 donne « Ca–on City » | Mac Roman est retenu comme hypothèse de travail car il donne « Cañon City », sans que l'encodage exact puisse être identifié de façon unique avec ce seul octet |
| Sauvegarde | Fichier réencodé en UTF-8 | `to_csv(..., encoding='utf-8', index=False)` |

**À retenir :** décoder sans erreur ne signifie pas décoder correctement. `chardet` fournit une hypothèse ; la vérification du contenu est indispensable.

And that's it for today! We didn't do quite as much coding, but  take my word for it: if you don't have the right tools, figuring out what encoding a file is in can be a huge time sink. If you have any questions, be sure to post them in the comments below or [on the forums](https://www.kaggle.com/questions-and-answers). 

Remember that your notebook is private by default, and in order to share it with other people or ask for help with it, you'll need to make it public. First, you'll need to save a version of your notebook that shows your current work by hitting the "Commit & Run" button. (Your work is saved automatically, but versioning your work lets you go back and look at what it was like at the point you saved it. It also lets you share a nice compiled notebook instead of just the raw code.) Then, once your notebook is finished running, you can go to the Settings tab in the panel to the left (you may have to expand it by hitting the [<] button next to the "Commit & Run" button) and setting the "Visibility" dropdown to "Public".

# More practice!
___

Check out [this dataset of files in different character encodings](https://www.kaggle.com/rtatman/character-encoding-examples). Can you read in all the files with their original encodings and them save them out as UTF-8 files?

If you have a file that's in UTF-8 but has just a couple of weird-looking characters in it, you can try out the [ftfy module](https://ftfy.readthedocs.io/en/latest/#) and see if it helps. 